# ParseCode.ipynb

A Jupyter Notebook hosting the backend code for the ClassTrack project.

Accepts PDFs, DOCXs, or plaintext via a Flask + Ngrok API. Then, returns a JSON object with relevant syllabus details for the frontend's Event Generation functionality.

This backend code is primarily developed by by Megu Kanzawa and Ethan Sychangco, with support from the rest of the ClassTrack team (Erin Schmidt and Josh Kindarara).

To see the version history, visit the Jupyter Notebook on [Google Colab](https://colab.research.google.com/drive/1PcVbO-RHdMwnMFIvtQuyIXu-9aeItSHS?usp=sharing).

# Credits

## Flask Setup in Google Colab

Template: [pyngrok Integration Examples](https://pyngrok.readthedocs.io/en/latest/integrations.html#google-colaboratory)

## Setting up Google Colab secrets

Guide: [How to use Secrets in Google Colab](https://medium.com/@parthdasawant/how-to-use-secrets-in-google-colab-450c38e3ec75)



# 1. Set Up Tokens

## IMPORTANT: MAKE SURE YOU HAVE ADDED ETHAN'S NGROK AND MEGU'S GEMINI TOKENS AS COLAB SECRETS

In [ ]:
from google.colab import userdata

# ================= #
# Flask Token Setup #
# ================= #

ng_token = userdata.get('NGROK_AUTHTOKEN')
# Token is under Ethan's ngrok account

# ========================= #
# Google Gemini Token Setup #
# ========================= #

gemini_token = userdata.get('GOOGLE_API_KEY')
# Token is under Megu's huggingface account

print("Tokens loaded.")

# 2. Package Installations


In [ ]:
# Flask - backend server
print("-=-=-=-=- [IMPORTING FLASK] -=-=-=-=-")
!pip install -q flask
!pip install -q flask_cors

# Pyngrok - expose Flask to the web
print("-=-=-=-=- [IMPORTING PYNGROK] -=-=-=-=-")
!pip install -q pyngrok

# PDF Plumber - parse PDFs
print("-=-=-=-=- [IMPORTING PDFPLUMBER] -=-=-=-=-")
!pip install -q pdfplumber

# Google Gemini Model
print("-=-=-=-=- [IMPORTING GOOGLE GEMINI] -=-=-=-=-")
!pip install -q google-generativeai

# Python Docx - parse DOCX
print("-=-=-=-=- [IMPORTING PYTHONDOCX] -=-=-=-=-")
!pip install python-docx

print("Libraries imported.")

# 3. Initialize Gemini Model

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

model_config = genai.Client(api_key=gemini_token)

# Basic sanity check
response = model_config.models.generate_content(
    model="gemini-2.0-flash",
    contents=["Explain how AI works"],
    config=types.GenerateContentConfig(
        max_output_tokens=500,
        temperature=0.1
    )
)
print(response.text)

# 4. MLModel Class

# Version 5 (Gemini)

Version History - viewable in Google Colab source file

Version 1 - Mistral

Version 2 - Mistral JSON

Version 3 - Mistral JSON Long

Version 4 - Gemini

Version 5 (current) - Gemini for PDF + DOCX + Text

In [ ]:
class MLModel:
    # Definition for a container class with an instance of our Gemini AI model

    # Note: previously stored a transformer-powered Mistral model, hence the
    # original name "ML." However, Gemini suits our needs better.

    MAX_INPUT_LEN = 3500
    PROMPT_PROLOGUE = f"""
### Instruction:
You will be given a syllabus text in varying formats (text, bullet points, or tables). Your tasks are:

1. **Course Info:**
- Extract the following fields from the syllabus:
    - **CourseCode**: This typically is a combination of the abbreviation (in ALL CAPS) and a number; ensure the abbreviation and number are separated by a space.
    - **CourseTitle**: The title of the course, including its course number and full title.
    - **Year**: The year of the course
    - **Quarter/Semester**: The quarter (Fall, Winter, Spring, Summer) or semester (Fall, Spring). It should only be one word.
    - **InstructorName**: The name of the instructor.
    - **InstructorEmail**: The instructor's email address (if provided, otherwise leave it empty).
    - **InstructorOffice**: The instructor's office, typically a combination of a building name and room number (if provided, otherwise leave it empty)
    - **GradingInformation**: A list of grading categories and their corresponding values, such as `{{"Category": "Homework", "Value": "10%"}}`.
        - Confirm that the percentages add up to each other: sometimes, a sub category will be given.
    - **Schedule**: List of class meeting times (Weekday(s), Start Time, End Time, Type, Location, Instructor).
        - For Weekdays, if Monday/Wednesday/Friday, write "M W F"; if Tuesday/Thursday, write "T Th"; if different, use a mixture of corresponding letters, with spaces in between. Monday is represented by "M", Tuesday by "T", Wednesday by "W", Thursday by "Th", and Friday by "F".
        - Location is typically a combination of a building name and room number; if on Zoom, use "Zoom" as the location; if not specified, leave it empty.
        - Type should be "Class" for regular classes, "Office Hour" for office hours, or "Lab" for lab sessions.
        - Instructor should be the name of the instructor or TA leading the class, lab, or office hour. If not specified, use the instructor's name from the course info (InstructorName).

2. **Schedule Information:**
- Extract the schedule information from any tabular format, lists, or bullet points in the syllabus.
- If presented in a table format, rows will be separated using "|". Parse each row top to bottom and left to right, making sure to handle any missing or incomplete information.
- For each event (such as classes, exams, etc.), return the following structure:
    ```json
    {{
        "EventName": "",
        "Date": "MM/DD/YYYY",
        "StartTime": "",
        "EndTime": "",
        "Weekday": "",
        "WeekNumber": "",
        "Reading": "",
        "Topics": "",
    }}
    ```
- If a week number or specific reading/topic is missing, assign topics based on the available content (e.g., class-by-class if larger than 10 topics).
- A week number should always be a number smaller than 12. If this is not true, it is not a week number, possibly the date.
- Give the above logic, if the week number is not specified, leave it empty.
- All midterm and final exam dates should be included in the schedule if a date is given and should always include the word "Exam" in the EventName.
    - If a midterm or final exam start time or end time is not provided, use the class start time and end time.
- If the event name is not specified but the reading/topic is available, assume it is a "Class" Event
- If the StartTime and EndTime is not provided, leave it empty.
- StartTime and EndTime must be formatted as H:MM AM or H:MM PM. Use this format even if the original text uses 24-hour time or abbreviations (like "0900" or "13:45"). Convert an times given in 24-hour format or other formats accordingly (e.g., 13:00 -> 1:00 PM, 1100 -> 11:00 AM).

3. **Important Dates:**
- Extract any event with a date (e.g., "First Class", "Exam", "Deadline to drop without W", "Final").
- Return in the following format:
    ```json
    {{
        "EventName": "",
        "Date": "MM/DD/YYYY",
        "Weekday": ""
    }}
    ```
- If the year is not specified, assume the year is **2024**.
- If the weekday is missing, infer it from the context of the date.
"""
    PROMPT_EPILOGUE = f"""
### Response:
Return a single valid JSON object with the following structure:

```json
{{
    "CourseCode": "",
    "CourseTitle": "",
    "Year": "",
    "Quarter/Semester": "",
    "InstructorName": "",
    "InstructorEmail": "",
    "GradingInformation": [
        {{"Category": "", "Value": ""}}
    ],
    "Schedule": [
        {{"Weekday": "", "Start Time": "", "End Time": "", "Type": "", "Location": "", "Instructor": ""}}
    ],
    "CourseSchedule": [
        {{
            "EventName": "",
            "Date": "MM/DD/YYYY",
            "StartTime": "",
            "EndTime": "",
            "Weekday": "",
            "WeekNumber": "",
            "Reading": "",
            "Topics": "",
        }}
    ],
    "ImportantDates": [
        {{
            "EventName": "",
            "Date": "MM/DD/YYYY",
            "Weekday": ""
        }}
    ]
}}
"""

    def __init__(self, model_config) -> None:
        # initalize loaded model
        self.gemini_client = model_config
        self.prompt = ""

    def set_syllabus_text(self, syllabus_text) -> None:
        # instance always starts with prompt prologue
        self.prompt = MLModel.PROMPT_PROLOGUE

        # add the rest of the prompt: parsed text and epilogue
        self.prompt += f"\n\n### Syllabus: \n{syllabus_text}"
        self.prompt += MLModel.PROMPT_EPILOGUE

    def invoke_model(self) -> str:
        # response
        response = self.gemini_client.models.generate_content(
            model="gemini-2.0-flash",
            contents=[self.prompt],
            config=types.GenerateContentConfig(
                max_output_tokens=4096,
                temperature=0.2
            )
        )

        print("MODEL RESPONSE:\n", response) if Parser.PRINT_DEBUG else None
        return response.text

# 5. JSONCleaner Class

In [ ]:
from datetime import datetime, timedelta
import re
import json

class JSONCleaner:
    # Defintion for a fully static utility class containing methods to
    # clean JSON objects generated by our AI.

    @staticmethod
    def clean_bad_symbols(output):
        # Clean fancy double quotes
        cleaned = output.replace('\u201c', "'")
        cleaned = cleaned.replace('\u201d', "'")

        # Clean fancy single quotes (NOTE: THEY GET WIPED FROM OUTPUT)
        cleaned = cleaned.replace('\u2019', '')
        cleaned = cleaned.replace('\u2018', '')

        # Clean ellipses
        cleaned = cleaned.replace('\u2026', '...')

        # Clean dashes
        cleaned = cleaned.replace('\u2013', ' - ')

        return cleaned

    @staticmethod
    def extract_json_from_output(output):
        # Remove the code block markers if present (backticks) before parsing
        clean_response = output.replace('```json', '').replace('```', '').strip()

        # Try to parse it as JSON
        try:
            return json.loads(clean_response)
        except json.JSONDecodeError:
            return {"raw_output": clean_response}

# 6. Parser Class (and children)

In [ ]:
from abc import ABC, abstractmethod

import pdfplumber

from docx import Document

import logging
# suppress only the specific warning from pdfminer.pdfpage
logging.getLogger("pdfminer.pdfpage").setLevel(logging.ERROR)

class Parser(ABC):
    # Definition for an abstract parser class, containing all necessary
    # classes to achieve syllabus text parsing functionality.

    # Parser is intended to be extended by more specialized parsers
    # such as TextParser, PDFParser, and DOCXParser.


    # Global debug variables and methods shared by all parsers
    TEST_FILE_NAME = None
    # if not none, override whatever file is sent by the request
    # and operate on this file instead
    PRINT_DEBUG = False
    # if true, all print statements with the ternary operator
    # "print() if PRINT_DEBUG else None" will output

    @staticmethod
    def set_print_debug(print_debug) -> None:
        Parser.PRINT_DEBUG = print_debug

    @staticmethod
    def set_test_PDF(name) -> None:
        Parser.TEST_FILE_NAME = name


    def __init__(self, model_obj, file_data, logger) -> None:
        self.model = model_obj
        self.logger = logger
        self.logger.report(state="init")

        if Parser.TEST_FILE_NAME is not None:
            # point to the forced file from the environment
            self.file_raw_content = Parser.TEST_FILE_NAME
        else:
            # point to request's FileStorage object
            self.file_raw_content = file_data

        self.full_text = ""
        self.model_output = ""
        self.clean_model_output = ""
        self.json = ""


    # Specialized parser child classes will override this
    @abstractmethod
    def parse_file(self):
        pass


    # Getters/setters
    def get_syllabus_text(self) -> str:
        return self.full_text

    def get_output(self) -> str:
        return self.model_output

    def get_json(self):
        return self.json


    # Connector functions to the contained model object
    def send_model_syllabus(self) -> None:
        self.logger.report(state="model-pre")
        self.model.set_syllabus_text(self.full_text)

    def invoke_model(self) -> None:
        self.model_output = self.model.invoke_model()
        self.logger.report(state="model-post")


    # Connector function to the JSONCleaner utility class
    def format_json(self) -> None:
        self.logger.report(state="json-pre")
        self.clean_model_output = JSONCleaner.clean_bad_symbols(self.model_output)
        self.json = JSONCleaner.extract_json_from_output(self.clean_model_output)
        self.logger.save_json(self.json)
        self.logger.report(state="json-post")


    # When finished, output statistics
    def end(self) -> None:
        self.logger.set_and_report_time()
        self.logger.report(state="end")


class TextParser(Parser):
    # Definition for a parser class optimized for plaintext
     # (i.e copied and pasted) syllabi.

    # Note: It simply returns the text it is given, and does not perform any
    # operations on it. Gemini is suited to handle long strings of plaintext,
    # and no extra operations from us are necessary.


    def __init__(self, model: MLModel, file_data, logger) -> None:
        super().__init__(model, file_data, logger) # file data is a string

    def parse_file(self):
        self.logger.report(state="syllabus-pre")

        print("EXTRACTED SECTIONS:\n", self.file_raw_content) if Parser.PRINT_DEBUG else None # DEBUG
        self.full_text = str(self.file_raw_content)

        self.logger.save_syllabus_text(self.full_text)
        self.logger.report(state="syllabus-post")
        return self.file_raw_content # just the string


class PDFParser(Parser):
    # Definition for a parser class optimized for PDF syllabi containing tables.

    def __init__(self, model: MLModel, file_data, logger) -> None:
        super().__init__(model, file_data, logger)

    def parse_file(self) -> str:
        self.logger.report(state="syllabus-pre")
        extracted_text = []

        with pdfplumber.open(self.file_raw_content) as pdf:
            for page in pdf.pages:
                words = page.extract_words(use_text_flow=True, keep_blank_chars=False)
                tables = page.extract_tables()

                # Group words into lines by their 'top' value
                lines_dict = {}
                for word in words:
                    top = round(word['top'], 1)
                    lines_dict.setdefault(top, []).append(word)

                # Convert lines_dict to ordered lines
                sorted_lines = []
                for top in sorted(lines_dict.keys()):
                    line_words = sorted(lines_dict[top], key=lambda w: w['x0'])  # left to right
                    line_text = " ".join(w['text'] for w in line_words).strip()
                    sorted_lines.append((top, line_text))

                # Add all the words from the sorted lines
                for _, line_text in sorted_lines:
                    if line_text:
                        extracted_text.append(line_text)

                # Add tables as well
                for table in tables:
                    for row in table:
                        row_text = " | ".join(
                            re.sub(r"\s*\n\s*", " ", str(cell or "")).strip() for cell in row
                        )
                        if row_text.strip():
                            extracted_text.append(row_text)

        # Combine everything into a single text output
        full_text = "\n".join(extracted_text)

        print("EXTRACTED SECTIONS:\n", full_text) if Parser.PRINT_DEBUG else None # DEBUG
        self.full_text = str(full_text)

        self.logger.report(state="syllabus-post")
        return self.full_text


class DOCXParser(Parser):
    # Definition for a parser class optimized for DOCX syllabi containing tables.

    def __init__(self, model: MLModel, file_data, logger) -> None:
        super().__init__(model, file_data, logger)

    def parse_file(self) -> str:
        self.logger.report(state="syllabus-pre")

        doc = Document(self.file_raw_content)
        full_text = "\n".join(para.text for para in doc.paragraphs)

        tables_data = []
        for table in doc.tables:
            table_data = []
            for row in table.rows:
                row_data = [cell.text.strip() for cell in row.cells]
                table_data.append(row_data)
            tables_data.append(table_data)

        full_text += ("Syllabus Tables: " + str(tables_data))

        print("EXTRACTED SECTIONS:\n", full_text) if Parser.PRINT_DEBUG else None # DEBUG
        self.full_text = str(full_text)

        self.logger.save_syllabus_text(self.full_text)
        self.logger.report(state="syllabus-post")
        return self.full_text

# 7. Logger Class

In [ ]:
import time

class RequestLogger:
    # Definition for a class designed to output status updates to the terminal
    # for requests to the Flask API.


    # static variable that controls how much is shown in terminal
    SAMPLE_MAX = 200


    def __init__(self, user_counter, be_quiet, req_path, file_data, file_type) -> None:
        # print(file_data)
        self.uid = user_counter
        self.be_quiet = be_quiet

        self.start_time = time.time()
        self.end_time = None

        self.req_path = req_path
        self.file_data = file_data
        self.file_type = file_type

        self.full_text = ""
        self.json = ""

        # string prepended to terminal outputs
        self.id = (str(self.uid) + "-" + self.file_type)


    def report(self, state: str) -> None:
        if self.be_quiet:
            return

        match state:
            case 'init':
                print(f"[{self.id}] [INCOMING REQUEST] - ID {self.uid}, to {self.req_path}, payload {self.file_data}")

            case 'syllabus-pre':
                print(f"[{self.id}] Extracting Syllabus...")

            case 'syllabus-post':
                print(f"[{self.id}] Syllabus successfully parsed.")
                print(f"[{self.id}] Syllabus text length:\n", len(self.full_text), "chars")
                print(f"[{self.id}] Syllabus text sample (first {RequestLogger.SAMPLE_MAX}):\n",
                        self.full_text[:RequestLogger.SAMPLE_MAX])

                print("\n--- End syllabus sample ---\n")
            case 'model-pre':
                print(f"[{self.id}] Prompting model...")

            case 'model-post':
                print(f"[{self.id}] Model output complete.")

            case 'json-pre':
                print(f"[{self.id}] Generating JSON...")

            case 'json-post':
                print(f"[{self.id}] JSON generated")
                print(f"[{self.id}] JSON text length:\n", len(self.json), "chars")
                print(f"[{self.id}] JSON text sample (first {RequestLogger.SAMPLE_MAX}):\n",
                        self.json[:RequestLogger.SAMPLE_MAX])
                print("\n--- End JSON sample ---\n")

            case 'end':
                print(f"[{self.id}] Request completed.")

            case _:
                # Handle default case
                print("WARNING: Logger entered default case")


    # save functions
    def save_syllabus_text(self, text) -> None:
        self.full_text = text

    def save_json(self, json) -> None:
        self.json = str(json)

    def set_and_report_time(self) -> None:
        self.end_time = time.time()
        print(f"[{self.id}] Time elapsed for request:\n",
            (self.end_time - self.start_time), "seconds")


# 8. Run Flask Server!

## NOTES FOR FLASK
1. Upon running this code, flask server should be set up like the following:

```
 * ngrok tunnel "NgrokTunnel: "https://starfish-calm-burro.ngrok-free.app" -> "http://localhost:5000"

```

2. To access the APIs we write here, write "/{path}" after the ngrok-free.app URL
    - There are two valid paths: `parsetext` and `parsefile`.
3. Errors will logged in the python output if something went wrong when the API was called
  - Nothing is shown on the site!!! just an error code...

## Testing - Force a File

In [ ]:
# FLASK TESTING
from google.colab import files

print("-=-=-=-=- [UPLOAD YOUR TEST PDF FILE] -=-=-=-=-")
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]

Parser.set_print_debug(True) # controls parser verbosity!
Parser.set_test_PDF(pdf_filename)

# Testing - Turn Off Forced File

In [ ]:
Parser.set_test_PDF(None)

# Testing - Output API Results

In [ ]:
Parser.set_print_debug(True)

# Testing - Don't Output API Results

In [ ]:
Parser.set_print_debug(False)

## Main Code

In [ ]:
import os

from flask import Flask, request
from pyngrok import ngrok, conf

import mimetypes
from werkzeug.datastructures import FileStorage

from flask_cors import CORS

from google.colab import files


# Driver code for our backend server.

# Accepts POST requests containing files or text to the free domain URL,
# returns a JSON object for use in the frontend logic.


app = Flask(__name__)
port = 5000

config = conf.get_default()
config.auth_token = ng_token # env variable from setup code block
# authtoken grabbed from "https://dashboard.ngrok.com/get-started/your-authtoken"

url = "starfish-calm-burro.ngrok-free.app" # Ethan's free domain


# Open a ngrok tunnel to the HTTP server
print("-=-=-=-=- [STARTING FLASK SERVER] -=-=-=-=-")
p_url = ngrok.connect(port, domain=url, pyngrok_config=config)
print(f" * ngrok tunnel \"{p_url}") # \" -> \"http://127.0.0.1:{port}\"")

# Update any base URLs to use the public ngrok URL
app.config["BASE_URL"] = p_url


# API Setup
userCounter = 1 # counter to identify requests in console

# Create logger object with proper user ID
def init_logger(be_quiet, path, file_data, file_type):
    global userCounter
    thisUser = userCounter
    userCounter += 1
    return RequestLogger(thisUser, be_quiet, path, file_data, file_type)

# File type checking to create the correct type of parser
def check_file_type_mimetype(file: FileStorage):
    content_type = file.content_type
    print("File Type Check:", content_type)

    if not content_type:
        return None

    if content_type == 'application/pdf':
        return 'pdf'
    elif content_type == 'application/vnd.openxmlformats-officedocument.wordprocessingml.document':
        return 'docx'
    else:
        return None


# Define Flask routes
@app.route("/")
def index():
    return "ClassTrack says hi! Please interface at /parsetext or /parsefile!"

@app.route("/parsefile", methods=['POST'])
def getJSONFromFile():

    # JavaScript front-end call to this code
    #
    # const uploadFile = async (file: File) => {
    #   ...
    #   const formData = new FormData();
    #   formData.append('file', file);
    #
    #   const response = fetch('https://starfish-calm-burro.ngrok-free.app/parsefile', {
    #       method: 'POST',
    #       body: formData
    #   });
    #   ...
    # }


    # Get syllabus PDF file
    file_data = request.files["file"]

    # Check filetype
    file_type = check_file_type_mimetype(file_data)


    # Initialize objects
    logger = init_logger(be_quiet=False,
                         path="parsefile",
                         file_data=file_data,
                         file_type=file_type)
    model_obj = MLModel(model_config)

    # select correct logger type
    if file_type is None:
        return "Invalid file type. Please upload a PDF or DOCX file.", 400
    elif file_type == "pdf":
        parser = PDFParser(model_obj, file_data, logger)
    elif file_type == "docx":
        parser = DOCXParser(model_obj, file_data, logger)


    # Perform parsing tasks
    parser.parse_file()

    parser.send_model_syllabus()
    parser.invoke_model()

    parser.format_json()


    # Complete Request!
    parser.end()
    return parser.get_json()


@app.route("/parsetext", methods=['POST'])
def getJSONFromText():

    # JavaScript front-end call to this code
    #
    # const uploadText = async (text: string) => {
    #   ...
    #   const response = fetch('https://starfish-calm-burro.ngrok-free.app/parsetext', {
    #       method: 'POST',
    #       body: text,
    #       headers: {
    #           'Content-Type': 'text/plain',
    #       }
    #   });
    #   ...
    # }

    # Get syllabus text data
    text_data = request.get_data(as_text=True)


    # Initialize objects
    logger = init_logger(be_quiet=False,
                         path="parsetext",
                         file_data="Text",
                         file_type="txt")
    model_obj = MLModel(model_config)

    parser = TextParser(model_obj, text_data, logger)


    # Perform parsing tasks
    parser.parse_file()

    parser.send_model_syllabus()
    parser.invoke_model()

    parser.format_json()


    # Complete Request!
    parser.end()
    return parser.get_json()


CORS(app)

app.run(port=port) # go!